In [2]:
#Librerías
import numpy as np
import pandas as pd

**Etapa 3 - Limpieza y Transformación de datos**

In [21]:
# -----------------------------------------------------------------------------
# Carga de datos originales y diccionario
# -----------------------------------------------------------------------------
df_raw = pd.read_csv("REG02_EPHC_ANUAL_2025.csv", sep=';', low_memory=False)
diccionario = pd.read_excel("diccionario_EPHC_ANUAL_2025.xls")

df = df_raw.copy()
print(f"Dimensiones iniciales: {df.shape[0]} filas, {df.shape[1]} columnas")

Dimensiones iniciales: 55934 filas, 213 columnas


In [9]:
# -----------------------------------------------------------------------------
# 1. Creación de Clave Primaria y verificación de duplicados
# -----------------------------------------------------------------------------
df["ID_PERSONA"] = (
    df["UPM"].astype(str)
    + "_"
    + df["NVIVI"].astype(str)
    + "_"
    + df["NHOGA"].astype(str)
    + "_"
    + df["L02"].astype(str)
)

duplicados_exactos = df.duplicated().sum()
duplicados_id = df.duplicated(subset=["ID_PERSONA"]).sum()

print(f"Duplicados exactos: {duplicados_exactos}")
print(f"Duplicados por ID_PERSONA: {duplicados_id}")
assert duplicados_id == 0, "Error: Existen claves primarias duplicadas."

Duplicados exactos: 0
Duplicados por ID_PERSONA: 0


In [10]:
# -----------------------------------------------------------------------------
# 2. Reemplazo de códigos de falta de respuesta (Valores centinela)
# -----------------------------------------------------------------------------
# Limpieza de valores como 999999999 o 99 en variables de ingreso y educación
centinelas_ingreso = [999999999, 99999999, 999999]
centinelas_educ = [99]

if "INGPRINC" in df.columns:
    df["INGPRINC"] = df["INGPRINC"].replace(centinelas_ingreso, np.nan)

if "ED01" in df.columns:
    df["ED01"] = df["ED01"].replace(centinelas_educ, np.nan)

In [15]:
# -----------------------------------------------------------------------------
# 3. Normalización de tipos de datos y formatos de texto
# -----------------------------------------------------------------------------
cols_categoricas = ["AREA", "P02_SEXO", "P06"]
for col in cols_categoricas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Normalizar columnas de ingreso a float64, y ED01 (años de estudio)
cols_num = ["INGPRINC", "INGTOT", "FEX", "ED01"]
for col in cols_num:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [16]:
# -----------------------------------------------------------------------------
# 4. Tratamiento de inconsistencias lógicas e imputación
# -----------------------------------------------------------------------------
# Inconsistencia: Escolaridad > Edad - 5
if "ED01" in df.columns and "P06" in df.columns:
    mask_inconsistente = df["ED01"] > (df["P06"] - 5)
    print(
        f"Registros con inconsistencia Edad vs Escolaridad: {mask_inconsistente.sum()}"
    )
    df.loc[mask_inconsistente, "ED01"] = np.maximum(
        0, df.loc[mask_inconsistente, "P06"] - 6
    )

# Imputación de ED01 (Años de estudio) por la mediana del grupo de edad
if "ED01" in df.columns:
    df["ED01"] = df.groupby(pd.cut(df["P06"], bins=[0, 14, 24, 49, 64, 100]))[
        "ED01"
    ].transform(lambda x: x.fillna(x.median()))

Registros con inconsistencia Edad vs Escolaridad: 27714


/tmp/ipykernel_1532/1520641422.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["ED01"] = df.groupby(pd.cut(df["P06"], bins=[0, 14, 24, 49, 64, 100]))[


In [18]:
# -----------------------------------------------------------------------------
# 5. Creación de variables derivadas
# -----------------------------------------------------------------------------
# Mapeo y conversión numérica de ingresos desde e01aimde / E01A y ipcm
if "e01aimde" in df.columns:
    df["INGPRINC"] = pd.to_numeric(
        df["e01aimde"].astype(str).str.replace(",", ".").str.strip(),
        errors="coerce",
    ).fillna(0)
elif "E01A" in df.columns:
    df["INGPRINC"] = pd.to_numeric(
        df["E01A"].astype(str).str.replace(",", ".").str.strip(),
        errors="coerce",
    ).fillna(0)

if "ipcm" in df.columns:
    df["INGTOT"] = pd.to_numeric(
        df["ipcm"].astype(str).str.replace(",", ".").str.strip(),
        errors="coerce",
    ).fillna(0)

# Población en Edad de Trabajar (PET >= 15 años usando P02)
df["PET"] = np.where(df["P02"] >= 15, 1, 0)


# Recodificación del Nivel Educativo (usando añoest o ED01)
def recodificar_educacion(row):
    col_edu = "añoest" if "añoest" in row else "ED01"
    anos = row.get(col_edu, np.nan)
    try:
        anos = float(str(anos).replace(",", "."))
    except (ValueError, TypeError):
        anos = np.nan

    if pd.isna(anos):
        return "Sin Instrucción"
    elif anos <= 2:
        return "1. Sin Instrucción / Inicial"
    elif 3 <= anos <= 9:
        return "2. Educación Básica"
    elif 10 <= anos <= 12:
        return "3. Educación Media"
    else:
        return "4. Superior / Universitaria"


df["NIVEL_EDUCATIVO_REC"] = df.apply(recodificar_educacion, axis=1)

# Categorización por Tramos de Edad (usando P02)
df["TRAMO_EDAD"] = pd.cut(
    df["P02"],
    bins=[-1, 14, 24, 49, 64, 120],
    labels=["Menores (<15)", "15-24", "25-49", "50-64", "65+"],
)

# Variable de Informalidad Laboral (1 = Informal, 2 = Formal según INE)
if "informalidad" in df.columns:
    df["INFORMAL_LABORAL"] = np.where(df["informalidad"] == 1, 1, 0)

# Subconjunto de trabajo: PEA Ocupada con ingreso laboral válido (> 0)
df_pea_ocupada = df[(df["PET"] == 1) & (df["INGPRINC"] > 0)].copy()

# Transformación logarítmica del ingreso principal
df_pea_ocupada["LN_INGPRINC"] = np.log(df_pea_ocupada["INGPRINC"])

In [19]:
# -----------------------------------------------------------------------------
# 6. Auditoría final de dataset transformado
# -----------------------------------------------------------------------------
print("\n--- RESUMEN DE PROCESAMIENTO ---")
print(f"Total registros dataset general transformado: {len(df)}")
print(f"Total registros submuestra PEA Ocupada con ingresos: {len(df_pea_ocupada)}")
print("\nDistribución por Nivel Educativo Recodificado (PEA Ocupada):")
print(df_pea_ocupada["NIVEL_EDUCATIVO_REC"].value_counts(normalize=True) * 100)


--- RESUMEN DE PROCESAMIENTO ---
Total registros dataset general transformado: 55934
Total registros submuestra PEA Ocupada con ingresos: 26459

Distribución por Nivel Educativo Recodificado (PEA Ocupada):
NIVEL_EDUCATIVO_REC
2. Educación Básica             38.576666
4. Superior / Universitaria     29.929325
3. Educación Media              27.450017
1. Sin Instrucción / Inicial     4.043993
Name: proportion, dtype: float64


In [20]:
# Exportar datasets limpios para la Etapa 4 (EDA y Análisis Inferencial)
df.to_csv("REG02_EPHC_ANUAL_2025_CLEAN_FULL.csv", index=False)
df_pea_ocupada.to_csv("REG02_EPHC_ANUAL_2025_CLEAN_PEA.csv", index=False)
print("\n Archivos limpios generados exitosamente.")


 Archivos limpios generados exitosamente.
